![Universidad Iberoamericana del Ecuador](https://raw.githubusercontent.com/andresrubiop/ml-unibe-estudiantes/main/utils/logo-unibe-color.png)

<div style="background:#004890;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0;border-bottom:4px solid #FCC000"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Machine Learning · Sesión 1 · Actividad calificada · aprendizaje autónomo</div><div style="font-size:22px;font-weight:700;margin-top:4px">Deber 1 · Comparación honesta de clasificadores en Fashion-MNIST</div><div style="font-size:12px;opacity:.9;margin-top:4px">Posgrado · Maestría en Big Data y Ciencia de Datos · UNIB.E</div></div>

| | |
|---|---|
| **Qué entregas** | **Este notebook ejecutado** (`.ipynb`), renombrado `D1_Apellido_Nombre.ipynb`, en el aula virtual. No hay que subir nada más. |
| **Fecha** | **Jueves 19 de noviembre de 2026** |
| **Tiempo estimado** | 3 a 4 horas (los modelos tardan entre 5 y 10 minutos en total en Colab) |
| **Peso** | 1/3 de los 2 puntos de aprendizaje autónomo · rúbrica al final de este notebook y en el aula |
| **Teoría** | Manual M1 §4–§12 · laboratorios L1.2 (seis clasificadores) y L1.3 (evaluar sin engañarse) |
| **Uso de IA** | **Permitido sin límite**: Claude Code, Codex, ChatGPT, Gemini en Colab, Copilot u otra herramienta, siempre que lo **declares** en la parte 6. |
| **Datos** | **Fashion-MNIST**: 70 000 fotos de 28 × 28 de 10 tipos de prendas (Xiao y colaboradores, 2017). La celda de carga ya está hecha. |

**Qué vas a hacer.** En L1.2 comparaste seis clasificadores en MNIST. Ahora llevas esa comparación a un problema más difícil y lo haces **con honestidad**: separas la prueba antes de todo, eliges los hiperparámetros con validación cruzada y miras la prueba **una sola vez**.

**Partes:** 0 · cómo trabajar → 1 · configuración → 2 · datos (hecho) → 3 · cinco clasificadores con validación cruzada → 4 · evaluación final y matriz de confusión → 5 · detector «¿es una camisa?» y curva ROC → 6 · desafío con IA y declaración → 7 · recomendación → lista de control.

## 0 · Cómo trabajar este notebook

1. **Guarda tu copia.** En Colab: *Archivo → Guardar una copia en Drive*. Trabaja siempre sobre esa copia: el original no guarda cambios.
2. **Ejecuta de arriba hacia abajo** (*Entorno de ejecución → Ejecutar todas*, o `Shift + Enter` celda por celda). Las celdas ya resueltas preparan los datos y las funciones de ayuda.
3. **Completa las celdas marcadas `# TODO`.** Cada parte explica qué se pide y trae una pista. Las celdas **✅ Autoverificación** usan `assert`: si no muestran error, vas bien.
4. **Responde en las celdas de texto** marcadas ✍️ (doble clic para editarlas). Escribe con tus palabras: la nota depende de que puedas explicar cada decisión.
5. **Antes de entregar:** *Entorno de ejecución → Reiniciar y ejecutar todas*. Si todo corre sin errores, descarga con *Archivo → Descargar → Descargar .ipynb*, renómbralo `D1_Apellido_Nombre.ipynb` y súbelo al aula virtual.

> **Sin Colab:** también funciona en JupyterLab local con el entorno del curso (`uv sync` en el repositorio y `uv run jupyter lab`).

## 1 · Configuración

Bibliotecas, semilla y estilo de las gráficas. No necesitas cambiar nada.

In [ ]:
import gzip
import os
import time
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026                     # semilla fija: tus resultados deben poder repetirse
UNIBE = {"azul": "#004890", "naranja": "#C2410C", "dorado": "#A87900", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UNIBE.values())), "axes.titlecolor": UNIBE["azul"],
    "axes.titleweight": "bold", "axes.edgecolor": UNIBE["gris"], "axes.grid": True, "grid.color": "#E8EDF2",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
CACHE = os.path.expanduser("~/.cache/ml-unibe")
URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
print(f"scikit-learn {sklearn.__version__} · NumPy {np.__version__} · semilla {SEED}")

## 2 · Los datos (ya resuelto)

Fashion-MNIST viene en cuatro archivos comprimidos (imágenes y etiquetas, entrenamiento y prueba) del repositorio público de datasets de Keras. La función `descargar()` los baja **una sola vez** (30 MB) y los guarda en caché; `cargar_fashion()` los convierte en arreglos de NumPy.

Para que todo corra en minutos usamos una **muestra fija**: 10 000 imágenes de entrenamiento y 2 000 de prueba. La prueba sale del archivo de prueba original, así que queda **separada desde el principio**: no la vuelves a tocar hasta la parte 4.

<details><summary><b>Ayuda: otras formas de cargar Fashion-MNIST</b></summary>

- **Con scikit-learn (OpenML):** `from sklearn.datasets import fetch_openml` y `X, y = fetch_openml("Fashion-MNIST", version=1, return_X_y=True, as_frame=False)`. Trae las 70 000 imágenes juntas (las primeras 60 000 son de entrenamiento).
- **Con PyTorch:** `torchvision.datasets.FashionMNIST(root="datos", train=True, download=True)`.
- **Sin conexión estable:** descarga los cuatro archivos `.gz` una vez, súbelos a tu Google Drive y cambia `CACHE` por la carpeta de Drive (ver la ayuda de Drive en el aula).
</details>

In [ ]:
def descargar(nombre):
    """Descarga un archivo del repositorio público de datasets de Keras (una sola vez)."""
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, nombre)
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(URL + nombre, ruta)
    return ruta


def cargar_fashion():
    """Fashion-MNIST: 60 000 imágenes de entrenamiento y 10 000 de prueba, 28 × 28 píxeles con valores de 0 a 255."""
    def imagenes(nombre):
        with gzip.open(descargar(nombre)) as f:
            return np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)

    def etiquetas(nombre):
        with gzip.open(descargar(nombre)) as f:
            return np.frombuffer(f.read(), np.uint8, offset=8)
    return (imagenes("train-images-idx3-ubyte.gz"), etiquetas("train-labels-idx1-ubyte.gz"),
            imagenes("t10k-images-idx3-ubyte.gz"), etiquetas("t10k-labels-idx1-ubyte.gz"))


PRENDAS = ["camiseta", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "botín"]
Xtr_full, ytr_full, Xte_full, yte_full = cargar_fashion()

rng = np.random.default_rng(SEED)
idx_tr = rng.choice(len(Xtr_full), 10_000, replace=False)
idx_te = rng.choice(len(Xte_full), 2_000, replace=False)
X_train = Xtr_full[idx_tr].reshape(-1, 784) / 255.0      # cada imagen es una fila de 784 píxeles entre 0 y 1
y_train = ytr_full[idx_tr]
X_test = Xte_full[idx_te].reshape(-1, 784) / 255.0       # prueba: separada desde el principio
y_test = yte_full[idx_te]
print("entrenamiento:", X_train.shape, "· prueba:", X_test.shape)
print("imágenes por clase en entrenamiento:", np.bincount(y_train))

fig, axs = plt.subplots(2, 10, figsize=(11, 2.6))
for k in range(10):
    for fila in range(2):
        i = int(np.flatnonzero(y_train == k)[fila])
        axs[fila, k].imshow(X_train[i].reshape(28, 28), cmap="Greys")
        axs[fila, k].axis("off")
    axs[0, k].set_title(PRENDAS[k], fontsize=8)
plt.show()

**Baseline.** Las 10 clases están balanceadas, así que un modelo que siempre responde la clase más frecuente acierta cerca del 10 %. Cualquier modelo tuyo debe superar con claridad esa cifra.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Baseline (siempre la clase más frecuente): exactitud en entrenamiento = {baseline.score(X_train, y_train):.3f}")

## 3 · Cinco clasificadores con validación cruzada

Compara **al menos cinco** clasificadores: **regresión logística**, **árbol de decisión**, **SVM**, **k vecinos** y **uno generativo** (LDA, QDA o Naive Bayes). Para cada uno:

- arma un `Pipeline` (la preparación va **dentro** del pipeline para que la validación cruzada no tenga fugas);
- elige **al menos un hiperparámetro** con `GridSearchCV` y validación cruzada estratificada de **3 pliegues**, solo con `X_train`;
- guarda la exactitud de validación como **media ± desviación** y el tiempo.

**Pista de velocidad.** 784 píxeles hacen lentos a la SVM y a k vecinos. Un `PCA(n_components=50)` dentro del pipeline conserva casi toda la información y acelera mucho. Te dejamos dos modelos hechos como ejemplo: completa los otros tres.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

# Cada entrada: nombre → (pipeline, rejilla de hiperparámetros). Los nombres de la rejilla llevan el prefijo del paso: "modelo__C".
candidatos = {
    "Regresión logística": (
        Pipeline([("escala", StandardScaler()), ("pca", PCA(50, random_state=SEED)),
                  ("modelo", LogisticRegression(max_iter=1000))]),
        {"modelo__C": [0.01, 0.1, 1.0]},
    ),
    "Árbol de decisión": (
        Pipeline([("modelo", DecisionTreeClassifier(random_state=SEED))]),
        {"modelo__max_depth": [8, 12, 16]},
    ),
}
# TODO: agrega la SVM, k vecinos y un modelo generativo, cada uno con su rejilla.
candidatos["SVM (RBF)"] = (
    Pipeline([("escala", StandardScaler()), ("pca", PCA(50, random_state=SEED)), ("modelo", SVC(kernel="rbf"))]),
    {"modelo__C": [...]},                       # prueba, por ejemplo, 1 y 10
)
candidatos["k vecinos"] = (...)                 # KNeighborsClassifier; hiperparámetro: n_neighbors
candidatos["..."] = (...)                       # LDA, QDA o GaussianNB; ¿qué hiperparámetro tiene sentido?
print(len(candidatos), "candidatos:", ", ".join(candidatos))

Ahora ajusta cada búsqueda y arma la tabla de resultados. Esta celda tarda entre 3 y 8 minutos en Colab (la SVM es la más lenta).

In [ ]:
filas, mejores = [], {}
for nombre, (pipe, rejilla) in candidatos.items():
    t0 = time.time()
    # TODO: crea un GridSearchCV con el pipeline, la rejilla, cv=cv y n_jobs=-1, y ajústalo SOLO con X_train, y_train.
    busqueda = GridSearchCV(...).fit(...)
    i = busqueda.best_index_
    filas.append({"modelo": nombre, "mejor hiperparámetro": busqueda.best_params_,
                  "exactitud CV (media)": busqueda.cv_results_["mean_test_score"][i],
                  "± desviación": busqueda.cv_results_["std_test_score"][i],
                  "tiempo (s)": time.time() - t0})
    mejores[nombre] = busqueda.best_estimator_
    print(f"{nombre:22s} {filas[-1]['exactitud CV (media)']:.3f} ± {filas[-1]['± desviación']:.3f}  ({filas[-1]['tiempo (s)']:.0f} s)")

tabla = pd.DataFrame(filas).sort_values("exactitud CV (media)", ascending=False).reset_index(drop=True)
tabla.round(3)

**✅ Autoverificación de la parte 3**

In [ ]:
assert len(tabla) >= 5, "Compara al menos cinco clasificadores."
assert tabla["exactitud CV (media)"].max() > 0.80, "El mejor modelo debería superar el 80 % en validación: revisa tus pipelines."
assert (tabla["exactitud CV (media)"] > 0.10).all(), "Algún modelo no supera al baseline del 10 %."
print("✓ Parte 3 correcta")

✍️ **Pregunta 3.** ¿Qué modelo gana en validación y por cuánto? ¿Las diferencias entre los dos mejores son mayores que su desviación? ¿Cuál es el más lento y vale la pena?

*Tu respuesta aquí.*

## 4 · Evaluación final, una sola vez

Elige el **mejor modelo según la validación** (no según la prueba) y evalúalo **una sola vez** en `X_test`. Luego:

1. dibuja la **matriz de confusión de 10 × 10** (`ConfusionMatrixDisplay`);
2. encuentra las **dos confusiones más frecuentes** (la pareja real → predicha con más errores, fuera de la diagonal);
3. muestra **ejemplos** de imágenes mal clasificadas de esas dos confusiones (la función `ver_errores` ya está hecha).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix


def ver_errores(real, predicha, X, y, y_pred, n=6):
    """Muestra hasta n imágenes de la clase `real` que el modelo predijo como `predicha`."""
    idx = np.flatnonzero((y == real) & (y_pred == predicha))[:n]
    fig, axs = plt.subplots(1, max(len(idx), 1), figsize=(1.6 * max(len(idx), 1), 1.9), squeeze=False)
    for ax, i in zip(axs[0], idx):
        ax.imshow(X[i].reshape(28, 28), cmap="Greys"); ax.axis("off")
    fig.suptitle(f"real: {PRENDAS[real]} → predicha: {PRENDAS[predicha]} ({len(np.flatnonzero((y == real) & (y_pred == predicha)))} casos)", fontsize=9)
    plt.show()


# TODO: toma el mejor modelo de la validación, predice X_test UNA vez, calcula la exactitud y la matriz de confusión.
nombre_mejor = ...                      # pista: la primera fila de `tabla`
modelo_final = mejores[nombre_mejor]
y_pred = ...
exactitud_prueba = ...
mc = ...
print(f"{nombre_mejor}: exactitud en la prueba = {exactitud_prueba:.3f}")
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(mc, display_labels=PRENDAS).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=60)
plt.show()

In [ ]:
# TODO: encuentra las dos parejas (real, predicha) con más errores fuera de la diagonal y muéstralas con ver_errores.
sin_diagonal = mc.copy()
np.fill_diagonal(sin_diagonal, 0)
confusiones = [...]                     # lista de dos tuplas (real, predicha); pista: np.argsort y np.unravel_index
for real, predicha in confusiones:
    ver_errores(real, predicha, X_test, y_test, y_pred)

**✅ Autoverificación de la parte 4**

In [ ]:
assert mc.shape == (10, 10) and mc.sum() == len(y_test), "La matriz de confusión debe ser de 10 × 10 y sumar 2 000."
assert len(confusiones) == 2 and all(r != p for r, p in confusiones), "Necesitas dos confusiones fuera de la diagonal."
assert abs(exactitud_prueba - tabla.loc[0, "exactitud CV (media)"]) < 0.05, "La prueba y la validación difieren mucho: ¿elegiste el modelo por la prueba?"
print("✓ Parte 4 correcta")

✍️ **Pregunta 4.** ¿La exactitud en la prueba se parece a la de validación? ¿Qué prendas se confunden y por qué crees que pasa (forma, textura, resolución de 28 × 28)?

*Tu respuesta aquí.*

## 5 · Detector binario «¿es una camisa?» y curva ROC

Una tienda quiere revisar a mano las fotos dudosas antes de publicarlas como «camisa». Convierte el problema en binario (`y == 6`, camisa) y:

1. entrena un clasificador que entregue **probabilidades** o puntajes (por ejemplo, una regresión logística con el mismo pipeline);
2. dibuja la **curva ROC** en la prueba y calcula el **AUC**;
3. elige un **umbral** y di qué porcentaje de camisas detecta (recall) y cuántas no camisas manda a revisión.

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_auc_score

yb_train, yb_test = (y_train == 6).astype(int), (y_test == 6).astype(int)
print(f"camisas: {yb_train.mean():.1%} del entrenamiento")

# TODO: entrena el detector con X_train, yb_train y calcula `puntajes` = probabilidad de camisa en X_test.
detector = Pipeline([...]).fit(X_train, yb_train)       # pista: class_weight="balanced" ayuda con la clase minoritaria
puntajes = detector.predict_proba(X_test)[:, 1]
auc = roc_auc_score(yb_test, puntajes)
RocCurveDisplay.from_predictions(yb_test, puntajes, name=f"detector de camisas (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color=UNIBE["gris"], label="azar")
plt.legend(); plt.show()

In [ ]:
# TODO: prueba un umbral y calcula el recall de camisas y la tasa de falsos positivos.
umbral = ...                           # prueba varios valores entre 0.1 y 0.9
pred_b = (puntajes >= umbral).astype(int)
recall = pred_b[yb_test == 1].mean()
fpr = pred_b[yb_test == 0].mean()
print(f"umbral {umbral}: detecta {recall:.1%} de las camisas · manda a revisión {fpr:.1%} de las no camisas "
      f"({pred_b.sum()} fotos de {len(pred_b)})")

**✅ Autoverificación de la parte 5**

In [ ]:
assert len(puntajes) == len(y_test) and 0 <= puntajes.min() and puntajes.max() <= 1, "Los puntajes deben ser probabilidades de la prueba."
assert auc > 0.80, "El AUC debería superar 0.80."
print("✓ Parte 5 correcta")

✍️ **Pregunta 5.** ¿Qué umbral recomendarías a la tienda y por qué? Piensa en el costo de cada error: una camisa mal publicada frente a una persona revisando fotos que no hacía falta revisar.

*Tu respuesta aquí.*

## 6 · Desafío con IA y declaración de uso

En este módulo **se permite el uso total de la IA** (Claude Code, Codex, ChatGPT, Gemini, Copilot u otra) en cualquier parte del notebook, **siempre que lo declares** en la tabla de abajo. Además, este deber incluye un **desafío con IA** obligatorio (lo pide el sílabo). Elige **uno**:

- **Opción A (L1.2):** pide a un asistente de IA (Claude, ChatGPT, Gemini en Colab, Copilot…) que agregue a la comparación un sexto modelo que no hayas usado (por ejemplo, `RandomForestClassifier` o `HistGradientBoostingClassifier`) **dentro de la misma validación cruzada**.
- **Opción B (L1.3):** pide a la IA que revise tu notebook buscando **fugas de datos** o usos indebidos de la prueba, y que proponga cómo corregirlos.

Ejecuta lo que te propone, **verifica** los resultados (¿usa la misma `cv`? ¿toca la prueba? ¿las cifras tienen sentido?) y corrige lo que esté mal. Pon el código en la celda de abajo.

In [ ]:
# TODO: código del desafío con IA (ya verificado y corregido por ti).
...

✍️ **Declaración de uso de IA** (obligatoria, aunque no hayas usado IA):

| | |
|---|---|
| **Herramienta y versión** | *por ejemplo: Claude (Anthropic), noviembre de 2026* |
| **Para qué la usaste** | *desafío, depuración, redacción…* |
| **Prompts principales** | *copia los dos o tres más importantes* |
| **Qué verificaste tú** | *cómo comprobaste que el resultado era correcto* |
| **Qué corregiste** | *errores de la IA que encontraste y cómo los arreglaste* |

## 7 · Recomendación (≈ 150 palabras)

✍️ Escribe para alguien que decide, no para un programador: ¿qué modelo usarías y por qué? Apóyate en las cifras de este notebook (exactitud, tiempo y tipo de error) y menciona una limitación.

*Tu recomendación aquí.*

## Lista de control antes de entregar

- [ ] *Reiniciar y ejecutar todas* corre sin errores y todas las autoverificaciones muestran ✓.
- [ ] La prueba se usó **una sola vez**, en la parte 4 (y para el detector de la parte 5).
- [ ] Respondiste las preguntas 3, 4 y 5, la declaración de IA y la recomendación.
- [ ] El archivo se llama `D1_Apellido_Nombre.ipynb` y lo subiste al aula virtual antes del jueves 19/11.

**Rúbrica (10 puntos):** reproducibilidad 1.5 · cinco clasificadores 2 · validación cruzada sin tocar la prueba 2 · matriz de confusión y ROC bien interpretadas 2 · desafío con IA verificado y declarado 1.5 · recomendación argumentada 1.